In [1]:
using Distributed
addprocs(22)
@everywhere using Analytical, CSV, DataFrames, JLD2, ProgressMeter
PATH = "/home/jmurga/mkt/202004/"
Analytical.sourcePlotMapR(script="/home/jmurga/script.jl")

"/home/jmurga/mkt/202004/"

We have used R to get the Maximum A Posteriori following ABCreg examples.

In [2]:
adap = Analytical.parameters(N=1000,n=661)

Parsing data from Uricchio et al. 2019

In [4]:
analysisFolder = PATH * "rawData/summStat/tgp/wg_v1/"

"/home/jmurga/mkt/202004/rawData/summStat/tgp/wg/"

In [4]:
wg = Analytical.parseSfs(sample=661,data= PATH * "rawData/tgp/wg.tsv",dac=adap.dac)

([-0.80121, -0.62347, -0.58417, -0.40921, -0.22421, -0.23045, -0.06355, -0.9144, 0.36187], [151646.0; 34219.0; … ; 302.0; 890.0], [73191], [0.0008 95554.0 56092.0; 0.0015 20028.0 14191.0; … ; 0.9985 112.0 190.0; 0.9992 340.0 550.0], [32154, 41037])

Writting SFS and divergence file in new foder analysis

In [6]:
run(`mkdir -p $analysisFolder`);
CSV.write(analysisFolder * "/sfsWg.tsv",DataFrame(wg[4],[:f,:pi,:p0]),delim='\t');
CSV.write(analysisFolder * "/divWg.tsv",DataFrame(wg[5]',[:di,:d0]),delim='\t');

"/home/jmurga/mkt/202004/rawData/summStat/tgp/wg//divWg.tsv"

Bootstrapping data following polyDFE manual

In [5]:
summaryStatsFromRates(param=adap,sFile=sFile,dFile=dFile,rates=h5file,dac=dac,summstatSize=10^3,replicas=replicas,bootstrap=true,outputFolder=analysisFolder);

Estimating summary statistics

In [6]:
replicas = 100
@time summstat = Analytical.summaryStatsFromRates(param=adap,rates=h5file,divergence=d,sfs=sfs,dac=dac,summstatSize=10^3,replicas=replicas);

progress_pmap(w,summstat,@. analysisFolder * "/summstat_" * string(1:replicas) * ".tsv";progress=Progress(replicas, desc="Writting summaries "));

Performing inference

In [8]:
Analytical.ABCreg(analysis=analysisFolder,replicas=100,P=5,S=9,tol=0.002,workers=20,abcreg="/home/jmurga/ABCreg/src/reg",parallel=true);

Estimating MAP distribution

In [ ]:
Analytical.sourcePlotMapR(script="/home/jmurga/script.jl")

In [ ]:
wgm = Analytical.plotMap(analysis=analysisFolder,output = PATH * "results/abc/wg_v2_map.svg")
describe(wgm)

In [ ]:
CSV.write(PATH * "results/abc/wg_map.tsv",wgm,delim='\t')

# VIPs

## TGP rates v1

In [ ]:
h5file   = jldopen(PATH * "rawData/tgp_v1.jld2")
adap.dac = h5file["1000/661/dac"]

analysisFolder = PATH * "rawData/summStat/tgp/vips_v1/"
vips = Analytical.parseSfs(sample=661,data= PATH *"rawData/tgp/vips.tsv",dac=adap.dac)
run(`mkdir -p $analysisFolder`)
CSV.write(analysisFolder * "/sfsVips.tsv",DataFrame(vips[4],[:f,:pi,:p0]),delim='\t')
CSV.write(analysisFolder * "/divVips.tsv",DataFrame(vips[5]',[:di,:d0]),delim='\t')

@time summstat = Analytical.summaryStatsFromRates(param=adap,rates=h5file,analysisFolder=analysisFolder,summstatSize=10^5,replicas=100,bootstrap=true);

Analytical.ABCreg(analysis=analysisFolder,replicas=100,P=5,S=size(adap.dac),tol=0.01,workers=20,abcreg="/home/jmurga/ABCreg/src/reg",parallel=true);

vipsm = Analytical.plotMap(analysis=analysisFolder,output = PATH * "results/abc/tgp/testSummaries/vips_v1_map.svg");
describe(vipsm)

## TGP rates v2

In [ ]:
h5file   = jldopen(PATH * "rawData/tgp_v2.jld2")
adap.dac = h5file["1000/661/dac"]

analysisFolder = PATH * "rawData/summStat/tgp/vips_v2/"
vips = Analytical.parseSfs(sample=661,data= PATH *"rawData/tgp/vips.tsv",dac=adap.dac)
run(`mkdir -p $analysisFolder`)
CSV.write(analysisFolder * "/sfsVips.tsv",DataFrame(vips[4],[:f,:pi,:p0]),delim='\t')
CSV.write(analysisFolder * "/divVips.tsv",DataFrame(vips[5]',[:di,:d0]),delim='\t')

@time summstat = Analytical.summaryStatsFromRates(param=adap,rates=h5file,analysisFolder=analysisFolder,summstatSize=10^5,replicas=100,bootstrap=true);

Analytical.ABCreg(analysis=analysisFolder,replicas=100,P=5,S=size(adap.dac,1),tol=0.01,workers=20,abcreg="/home/jmurga/ABCreg/src/reg",parallel=true);

vipsm = Analytical.plotMap(analysis=analysisFolder,output = PATH * "results/abc/tgp/testSummaries/vips_v2_map.svg");
describe(vipsm)

## TGP rates v3

In [ ]:
h5file   = jldopen(PATH * "rawData/tgp_v3.jld2")
adap.dac = h5file["1000/661/dac"]

analysisFolder = PATH * "rawData/summStat/tgp/vips_v3/"
vips = Analytical.parseSfs(sample=661,data= PATH *"rawData/tgp/vips.tsv",dac=adap.dac)
run(`mkdir -p $analysisFolder`)
CSV.write(analysisFolder * "/sfsVips.tsv",DataFrame(vips[4],[:f,:pi,:p0]),delim='\t')
CSV.write(analysisFolder * "/divVips.tsv",DataFrame(vips[5]',[:di,:d0]),delim='\t')

@time summstat = Analytical.summaryStatsFromRates(param=adap,rates=h5file,analysisFolder=analysisFolder,summstatSize=10^5,replicas=100,bootstrap=true);

Analytical.ABCreg(analysis=analysisFolder,replicas=100,P=5,S=size(adap.dac,1),tol=0.01,workers=20,abcreg="/home/jmurga/ABCreg/src/reg",parallel=true);

vipsm = Analytical.plotMap(analysis=analysisFolder,output = PATH * "results/abc/tgp/testSummaries/vips_v3_map.svg");
describe(vipsm)

## TGP rates v4

In [ ]:
h5file   = jldopen(PATH * "rawData/tgp_v4.jld2")
adap.dac = h5file["1000/661/dac"]

analysisFolder = PATH * "rawData/summStat/tgp/vips_v4/"
vips = Analytical.parseSfs(sample=661,data= PATH *"rawData/tgp/vips.tsv",dac=adap.dac)
run(`mkdir -p $analysisFolder`)
CSV.write(analysisFolder * "/sfsVips.tsv",DataFrame(vips[4],[:f,:pi,:p0]),delim='\t')
CSV.write(analysisFolder * "/divVips.tsv",DataFrame(vips[5]',[:di,:d0]),delim='\t')

@time summstat = Analytical.summaryStatsFromRates(param=adap,rates=h5file,analysisFolder=analysisFolder,summstatSize=10^5,replicas=100,bootstrap=true);

Analytical.ABCreg(analysis=analysisFolder,replicas=100,P=5,S=size(adap.dac,1),tol=0.01,workers=20,abcreg="/home/jmurga/ABCreg/src/reg",parallel=true);

vipsm = Analytical.plotMap(analysis=analysisFolder,output = PATH * "results/abc/tgp/testSummaries/vips_v4_map.svg");
describe(vipsm)